In [ ]:
# Locate the project root and standard repository folders.
import os
from pathlib import Path

try:
    import google.colab  # type: ignore  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)


def find_project_root() -> Path:
    """Find the repository root from a notebook, script, local shell, or Colab."""
    try:
        here = Path(__file__).resolve().parent  # type: ignore[name-defined]
    except NameError:
        here = Path.cwd().resolve()

    candidates = [here, *here.parents, Path.cwd().resolve()]
    candidates.extend([
        Path("/content/1.C51-ML-Project"),
        Path("/content/drive/MyDrive/1.C51-ML-Project"),
        Path("/content/drive/MyDrive/Machine Learning Class Project"),
    ])

    seen = set()
    for cand in candidates:
        try:
            cand = cand.expanduser().resolve()
        except Exception:
            continue
        if cand in seen:
            continue
        seen.add(cand)
        if (cand / "README.md").exists() and (cand / "requirements.txt").exists():
            return cand
    return Path.cwd().resolve()


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
METADATA_DIR = DATA_DIR / "metadata"
PROCESSED_DIR = DATA_DIR / "processed"
CARBON_DIR = DATA_DIR / "carbon"
MODEL_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"

for directory in [RAW_DATA_DIR, PROCESSED_DIR, CARBON_DIR, MODEL_DIR, RESULTS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

# Backward-compatible alias for older cells/comments.
TEAM_DIR = PROJECT_ROOT

print("IN_COLAB =", IN_COLAB)
print("PROJECT_ROOT =", PROJECT_ROOT)
print("RAW_DATA_DIR =", RAW_DATA_DIR)
print("PROCESSED_DIR =", PROCESSED_DIR)
print("MODEL_DIR =", MODEL_DIR)
print("RESULTS_DIR =", RESULTS_DIR)


In [ ]:
# Kaggle credentials.
# Do NOT hard-code the token in this shared notebook.
# - Colab: add KAGGLE_API_TOKEN via "Secrets" (key icon on the left) and uncomment below,
#   or upload kaggle.json to ~/.kaggle/ once.
# - Local: set it in your shell env, or place kaggle.json at ~/.kaggle/kaggle.json.
#
# import os
# from google.colab import userdata  # Colab only
# os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")
#
# Note: this cell is optional -- the dataset is already cached in PROJECT_ROOT,
# so the download cell below will be skipped when the data already exists.


In [ ]:
# Locate the Smart Meters in London dataset.
# The full raw dataset is intentionally not tracked in Git; see data/README.md.
import os, sys, subprocess
from pathlib import Path

SMART_METER_DIR = RAW_DATA_DIR / "smart_meter_data"

if SMART_METER_DIR.exists() and any(SMART_METER_DIR.iterdir()):
    path = str(SMART_METER_DIR)
    print("Using cached Smart Meters data at:", path)
else:
    print("Smart Meters data not found locally; downloading via kagglehub...")
    try:
        import kagglehub
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "kagglehub"])
        import kagglehub

    os.environ["KAGGLEHUB_CACHE_DIR"] = str(RAW_DATA_DIR / "kagglehub_cache")
    path = kagglehub.dataset_download("jeanmidev/smart-meters-in-london")
    print("Downloaded to:", path)


In [ ]:
import os, sys, subprocess
from pathlib import Path

NESO_URL = "https://api.neso.energy/dataset/88313ae5-94e4-4ddc-a790-593554d8c6b9/resource/f93d1835-75bc-43e5-84ad-12472b180a98/download/df_fuel_ckan.csv"
NESO_OUT = CARBON_DIR / "neso_historic_generation_mix.csv"

if NESO_OUT.exists() and NESO_OUT.stat().st_size > 0:
    print("Using cached NESO file:", NESO_OUT)
else:
    print("NESO file not found locally; downloading...")
    try:
        import requests
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "requests"])
        import requests

    with requests.get(NESO_URL, stream=True, timeout=300) as r:
        r.raise_for_status()
        with open(NESO_OUT, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)
    print("NESO saved:", NESO_OUT)


### 1. Inspecting Smart Meters in London Dataset
Since this dataset contains multiple files, let's list the directory contents and read a representative file (e.g., `informations_households.csv`).

In [ ]:
import pandas as pd
import os

# List all files in the downloaded Kaggle path
print("Files in Smart Meters directory:")
for root, dirs, files in os.walk(path):
    for file in files:
        if file.endswith('.csv'):
            print(os.path.join(root, file))

# Load a sample file (e.g., household information)
sample_file_path = os.path.join(path, 'informations_households.csv')
if os.path.exists(sample_file_path):
    df_smart_meter = pd.read_csv(sample_file_path)
    print("\nSmart Meter (Households Info) Preview:")
    display(df_smart_meter.head())
else:
    print("\nCould not find informations_households.csv in the expected path.")

### 2. Inspecting NESO Generation Mix Dataset
Now we load the historic generation mix data downloaded from NESO.

In [ ]:
# Load the NESO generation data from data/carbon
df_generation = pd.read_csv(NESO_OUT)
print("Generation Mix Data Preview:")
display(df_generation.head())

print("\nColumns in Generation Mix:")
print(df_generation.columns.tolist())

### 3. Verify Repository Storage Structure
Inspect the repository layout so the raw data, processed features, models, and results folders are easy to locate.


In [ ]:
import os

def list_files(startpath):
    startpath = os.fspath(startpath)
    for root, dirs, files in os.walk(startpath):
        level = root.replace(startpath, '').count(os.sep)
        indent = ' ' * 4 * (level)
        print(f'{indent}{os.path.basename(root)}/')
        subindent = ' ' * 4 * (level + 1)
        for f in files[:5]:
            print(f'{subindent}{f}')
        if len(files) > 5:
            print(f'{subindent}... and {len(files)-5} more files')

print(f"Structure of {PROJECT_ROOT}:")
list_files(PROJECT_ROOT)

### 4. Backup Data to Google Drive
Since the data is currently in a temporary Kaggle cache, let's copy it to our shared Team Directory on Google Drive for persistence.

In [ ]:
import os, shutil
from pathlib import Path

destination_path = SMART_METER_DIR
src_path = Path(path)

if destination_path.resolve() == src_path.resolve():
    print(f"Data already lives in the repository data folder: {destination_path}")
elif destination_path.exists() and any(destination_path.iterdir()):
    print(f"Data already exists in repository data folder: {destination_path}")
else:
    print(f"Copying data from {src_path} to {destination_path}...")
    destination_path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(src_path, destination_path)
    print("Copy complete!")

SMART_METER_DIR = destination_path
print(f"\nContents of {destination_path}:")
print(os.listdir(destination_path))


## 5. Short-term Half-hourly Electricity Demand Forecasting


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

RANDOM_STATE = 42

SMART_DIR = SMART_METER_DIR
HH_DIR    = SMART_DIR / "halfhourly_dataset" / "halfhourly_dataset"
CACHE_DIR = PROCESSED_DIR
CACHE_DIR.mkdir(parents=True, exist_ok=True)

assert SMART_DIR.exists(), f"SMART_DIR missing: {SMART_DIR}"
assert HH_DIR.exists(),    f"HH_DIR missing:    {HH_DIR}"

print("SMART_DIR :", SMART_DIR)
print("HH_DIR    :", HH_DIR)
print("CACHE_DIR :", CACHE_DIR)
print("RANDOM_STATE =", RANDOM_STATE)

### 5.1 Stratified sampling + on-demand block loading

Pick 500 LCLids with `StratifiedShuffleSplit` on the `Acorn_grouped | stdorToU` key, then iterate only the halfhourly block files that contain them.

The `informations_households.csv` holds ~5566 households spread over 112 blocks (~50 households per block). Sampling ~9% of households means almost every block is hit — expect to read **80+ blocks**. We still get a ~24× I/O reduction vs. reading everything, because each block is filtered to the ~4–5 sampled households inside it.

Tiny strata (<5 households) get collapsed into a single "Other" bucket so the split doesn't choke on size=1 classes.

In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit

meta = pd.read_csv(SMART_DIR / "informations_households.csv")
print(f"Total households: {len(meta):,}  |  blocks on disk: {meta['file'].nunique()}")

meta["stratum"] = meta["Acorn_grouped"].astype(str) + "|" + meta["stdorToU"].astype(str)
counts = meta["stratum"].value_counts()
print("\nStratum sizes (top/bottom):")
print(counts.head(10).to_string())
print("  ...")
print(counts.tail(5).to_string())

tiny = counts[counts < 5].index.tolist()
if tiny:
    print(f"\nCollapsing {len(tiny)} strata with <5 members into 'Other': {tiny}")
    meta["stratum"] = meta["stratum"].where(~meta["stratum"].isin(tiny), "Other")

sss = StratifiedShuffleSplit(n_splits=1, train_size=500, random_state=RANDOM_STATE)
idx_train, _ = next(sss.split(meta, meta["stratum"]))
sampled = meta.iloc[idx_train].copy()
sampled_ids = set(sampled["LCLid"])
assert len(sampled_ids) == 500

print(f"\nSampled {len(sampled_ids)} households.")
print("Sampled stratum composition:")
print(sampled["stratum"].value_counts().to_string())

needed_blocks = sorted(sampled["file"].unique())
print(f"\nBlocks to read: {len(needed_blocks)} / {meta['file'].nunique()}")
print(f"Sampled LCLids per block: min={sampled['file'].value_counts().min()}, "
      f"median={int(sampled['file'].value_counts().median())}, "
      f"max={sampled['file'].value_counts().max()}")

import time
t0 = time.time()
chunks = []
for i, block in enumerate(needed_blocks, 1):
    fp = HH_DIR / f"{block}.csv"
    if not fp.exists():
        print(f"  WARN: {fp.name} missing — skipping")
        continue
    chunk = pd.read_csv(
        fp,
        usecols=["LCLid", "tstp", "energy(kWh/hh)"],
        dtype={"LCLid": "category", "tstp": "object", "energy(kWh/hh)": "object"},
    )
    chunk = chunk[chunk["LCLid"].astype(str).isin(sampled_ids)]
    chunks.append(chunk)
    if i % 20 == 0 or i == len(needed_blocks):
        rows = sum(len(c) for c in chunks)
        print(f"  [{i:3d}/{len(needed_blocks)}] rows so far: {rows:>12,}  elapsed {time.time()-t0:5.1f}s")

df_raw = pd.concat(chunks, ignore_index=True)
df_raw["LCLid"] = df_raw["LCLid"].astype(str)
del chunks

print(f"\nRaw combined shape: {df_raw.shape}")
print(f"Unique LCLids: {df_raw['LCLid'].nunique()} (expected {len(sampled_ids)})")
missing = sampled_ids - set(df_raw["LCLid"].unique())
if missing:
    print(f"WARN: {len(missing)} sampled LCLids absent from block files (possible empty/short series).")
    print("  example missing IDs:", list(missing)[:5])

df_raw.head()

### 5.2 Cleaning + timestamp normalization

Raw `tstp` values look like `2012-10-12 00:30:00.0000000` — **7 fractional digits**, which Python's `%f` format (officially ≤6) does not guarantee to parse. We slice to the first 19 characters and parse with `"%Y-%m-%d %H:%M:%S"` instead (all fractional parts are zero anyway).

Steps:

1. Parse `tstp` and cast `energy(kWh/hh)` to `float32` (whitespace/`Null`/empty → `NaN`).
2. Drop household-days with fewer than 46 of 48 half-hours.
3. Sort by `(LCLid, tstp)` and interpolate gaps within each household (`limit=4` half-hours = 2 h); drop rows still NaN.
4. Sanity-check timestamp range and the median number of days per household.

In [ ]:
print(f"Before cleaning: {df_raw.shape}")

if not pd.api.types.is_datetime64_any_dtype(df_raw["tstp"]):
    df_raw["tstp"] = pd.to_datetime(
        df_raw["tstp"].astype(str).str.slice(0, 19),
        format="%Y-%m-%d %H:%M:%S",
        errors="coerce",
        cache=True,
    )
n_bad_tstp = df_raw["tstp"].isna().sum()
print(f"tstp parse failures: {n_bad_tstp:,}")
assert n_bad_tstp == 0, "Unexpected tstp parse failures"

if "energy(kWh/hh)" in df_raw.columns:
    energy_str = df_raw["energy(kWh/hh)"].astype(str).str.strip()
    energy_str = energy_str.where(~energy_str.isin(["Null", "null", "NaN", "nan", "None", ""]), other=None)
    df_raw["energy"] = pd.to_numeric(energy_str, errors="coerce").astype("float32")
    df_raw = df_raw.drop(columns=["energy(kWh/hh)"])
elif "energy" not in df_raw.columns:
    raise KeyError("Neither 'energy(kWh/hh)' nor 'energy' present — did 5.1 run?")
print(f"Energy NaN (pre-interp): {df_raw['energy'].isna().sum():,}")
print(f"tstp range: {df_raw['tstp'].min()}  →  {df_raw['tstp'].max()}")

df_raw["date"] = df_raw["tstp"].dt.floor("D")
hh_count = df_raw.groupby(["LCLid", "date"])["energy"].transform("size")
before = len(df_raw)
df_raw = df_raw[hh_count >= 46].copy()
print(f"Dropped {before - len(df_raw):,} rows from household-days with <46 half-hours")

df_raw = df_raw.sort_values(["LCLid", "tstp"], ignore_index=True)
df_raw["energy"] = (
    df_raw.groupby("LCLid", observed=True)["energy"]
          .transform(lambda s: s.interpolate(method="linear", limit=4, limit_direction="both"))
          .astype("float32")
)
still_na = df_raw["energy"].isna().sum()
print(f"Post-interpolation NaN (dropping): {still_na:,}")
df_raw = df_raw.dropna(subset=["energy"]).reset_index(drop=True)

days_per_hh = df_raw.groupby("LCLid")["date"].nunique()
print(f"\nDays per household: min={days_per_hh.min()}, median={int(days_per_hh.median())}, max={days_per_hh.max()}")
print(f"tstp range after cleaning: {df_raw['tstp'].min()} → {df_raw['tstp'].max()}")
assert df_raw["tstp"].min() >= pd.Timestamp("2011-11-01"), "unexpected early tstp"
assert df_raw["tstp"].max() <= pd.Timestamp("2014-03-31"), "unexpected late tstp"
assert days_per_hh.median() >= 300, "median days-per-household < 300; sample unreliable"

df_clean = df_raw.drop(columns=["date"])
del df_raw

print(f"\ndf_clean shape: {df_clean.shape}")
print(f"memory: {df_clean.memory_usage(deep=True).sum()/1e6:,.1f} MB")
print(df_clean.dtypes.to_string())
df_clean.head()

### 5.3 Merge weather + bank holidays + household metadata

- **Weather** (`weather_hourly_darksky.csv`): hourly → 30-min via `ffill`, then derive
  `hdd = max(0, 15.5 − T)` and `cdd = max(0, T − 22)` (UK base temperatures). `precipType` NaN → `"none"`.
- **Holidays** (`uk_bank_holidays.csv`): date set → binary `is_holiday`. File is Latin-1 encoded (curly quotes in `Type`), so read with `encoding="latin1"`.
- **Metadata**: left-join `Acorn / Acorn_grouped / stdorToU` onto every row.

> **ToU caveat** — households with `stdorToU == "ToU"` joined dynamic-pricing trials from 2013-01,
> which introduces a distributional shift. We report errors bucketed by this flag in 5.8.

In [ ]:
WEATHER_NUM = ["temperature", "apparentTemperature", "humidity",
               "windSpeed", "pressure", "dewPoint", "visibility"]
WEATHER_CAT = ["icon", "precipType"]

weather = pd.read_csv(SMART_DIR / "weather_hourly_darksky.csv")
weather["time"] = pd.to_datetime(weather["time"], errors="coerce")
assert weather["time"].isna().sum() == 0, "weather time parse failures"
weather = (
    weather[["time", *WEATHER_NUM, *WEATHER_CAT]]
    .set_index("time")
    .sort_index()
)
print(f"Weather raw: {weather.shape}, range {weather.index.min()} → {weather.index.max()}")

weather_30 = weather.resample("30min").ffill()
weather_30["hdd"] = (15.5 - weather_30["temperature"]).clip(lower=0)
weather_30["cdd"] = (weather_30["temperature"] - 22.0).clip(lower=0)
weather_30["precipType"] = weather_30["precipType"].fillna("none")
for c in [*WEATHER_NUM, "hdd", "cdd"]:
    weather_30[c] = weather_30[c].astype("float32")
print(f"Weather 30-min: {weather_30.shape} (NaN per col: "
      f"{int(weather_30[WEATHER_NUM].isna().sum().sum())})")

hol = pd.read_csv(SMART_DIR / "uk_bank_holidays.csv", encoding="latin1")
hol_dates = pd.to_datetime(hol["Bank holidays"]).dt.date
hol_set = set(hol_dates)
hol_years = {d.year for d in hol_set}
print(f"\nHolidays: {len(hol_set)} dates across years {sorted(hol_years)}")
assert hol_years.issuperset({2012, 2013, 2014}), "Missing 2012–2014 bank holidays"

meta_cols = ["LCLid", "Acorn", "Acorn_grouped", "stdorToU"]
meta_mrg = meta[meta_cols].drop_duplicates("LCLid").copy()

df_feat = df_clean.merge(
    weather_30.reset_index().rename(columns={"time": "tstp"}),
    on="tstp", how="left",
)
df_feat = df_feat.merge(meta_mrg, on="LCLid", how="left")
df_feat["is_holiday"] = df_feat["tstp"].dt.date.isin(hol_set).astype("int8")

for c in ["icon", "precipType", "Acorn", "Acorn_grouped", "stdorToU"]:
    df_feat[c] = df_feat[c].astype("category")

weather_na = df_feat[WEATHER_NUM].isna().sum()
meta_na    = df_feat[["Acorn", "Acorn_grouped", "stdorToU"]].isna().sum().sum()
print(f"\nAfter merge: shape = {df_feat.shape}")
print(f"memory: {df_feat.memory_usage(deep=True).sum()/1e6:,.1f} MB")
print(f"Weather NaN after merge:\n{weather_na.to_string()}")
print(f"Meta NaN (should be 0): {meta_na}")
print(f"Holiday hit rate: {df_feat['is_holiday'].mean():.4f}")
print(f"ToU hit rate   : {(df_feat['stdorToU'] == 'ToU').mean():.4f}")

assert meta_na == 0, "Some rows have no household metadata"
assert weather_na.sum() < 0.01 * len(df_feat), "Too many weather NaNs (>1%)"

print("\n--- df_feat.dtypes ---")
print(df_feat.dtypes.to_string())
df_feat.head()

### 5.4 Feature engineering

Build the predictor columns on top of `df_feat`, in leak-safe order. We keep a single cleaned panel, then expose it in **two model views**:

- **Tabular view** for seasonal naive / Ridge / LightGBM: lagged load, rolling stats, calendar, weather, and static household metadata.
- **Sequence view** for the LSTM benchmark in 5.8: rolling windows of recent half-hours plus the same household metadata.

Steps:

1. **Drop tiny-history households** (`<30 days` of valid data). With 15 M rows, these are noise and waste memory.
2. **Set temporal cutoffs** (70 % / 85 % quantiles of `tstp`). These same cutoffs drive 5.5's split and are captured here so that `household_train_mean` uses exactly the train window.
3. **`household_train_mean`** — per-LCLid average of `energy` restricted to rows before `train_cutoff`, then broadcast to every row. Substitutes for an `LCLid` identity feature in LightGBM without leaking future info.
4. **Time features**: `hour / half_hour / dow / month / is_weekend` + cyclic `sin/cos` encodings for Ridge and LSTM.
5. **Lags**: `lag_1, lag_2, lag_48, lag_336` via within-group `shift()`.
6. **Rolling stats** over `lag_1` (the already-shifted series) so the window never sees `t`:
   `rolling_{mean,std}_1d` (window 48) and `rolling_{mean,std}_7d` (window 336).
7. Drop the first-week-per-household rows (where `lag_336` / `rolling_7d` are NaN).
8. Downcast any float64 that snuck in → float32, then save to parquet/pickle.


In [ ]:
import time, gc

df_feat = df_feat.sort_values(["LCLid", "tstp"], ignore_index=True)

hh_rows = df_feat.groupby("LCLid", observed=True).size()
keep_ids = hh_rows[hh_rows >= 30 * 46].index
dropped = hh_rows.size - len(keep_ids)
if dropped > 0:
    print(f"Dropping {dropped} households with <30 days of data "
          f"(rows: {int(hh_rows[hh_rows < 30*46].sum()):,})")
    df_feat = df_feat[df_feat["LCLid"].isin(keep_ids)].reset_index(drop=True)
print(f"Households retained: {df_feat['LCLid'].nunique()}  rows: {len(df_feat):,}")

train_cutoff = df_feat["tstp"].quantile(0.70)
val_cutoff   = df_feat["tstp"].quantile(0.85)
print(f"\ntrain cutoff (70%): {train_cutoff}")
print(f"val   cutoff (85%): {val_cutoff}")

_train_mask_5p4 = df_feat["tstp"] < train_cutoff
hh_mean = df_feat.loc[_train_mask_5p4].groupby("LCLid", observed=True)["energy"].mean()
df_feat["household_train_mean"] = df_feat["LCLid"].map(hh_mean).astype("float32")
n_missing_mean = df_feat["household_train_mean"].isna().sum()
if n_missing_mean:
    global_mean = float(df_feat.loc[_train_mask_5p4, "energy"].mean())
    print(f"WARN: {n_missing_mean:,} rows had no train-window mean; filling with global mean {global_mean:.4f}")
    df_feat["household_train_mean"] = df_feat["household_train_mean"].fillna(global_mean).astype("float32")
print(f"household_train_mean stats: "
      f"min={df_feat['household_train_mean'].min():.4f}, "
      f"median={df_feat['household_train_mean'].median():.4f}, "
      f"max={df_feat['household_train_mean'].max():.4f}")

ts = df_feat["tstp"]
df_feat["hour"]       = ts.dt.hour.astype("int8")
df_feat["half_hour"]  = (ts.dt.hour.astype("int16") * 2
                         + (ts.dt.minute == 30).astype("int8")).astype("int8")
df_feat["dow"]        = ts.dt.dayofweek.astype("int8")
df_feat["month"]      = ts.dt.month.astype("int8")
df_feat["is_weekend"] = (df_feat["dow"] >= 5).astype("int8")

df_feat["hh_sin"]  = np.sin(2 * np.pi * df_feat["half_hour"] / 48).astype("float32")
df_feat["hh_cos"]  = np.cos(2 * np.pi * df_feat["half_hour"] / 48).astype("float32")
df_feat["dow_sin"] = np.sin(2 * np.pi * df_feat["dow"]       / 7 ).astype("float32")
df_feat["dow_cos"] = np.cos(2 * np.pi * df_feat["dow"]       / 7 ).astype("float32")

print("\nBuilding lag features...")
t0 = time.time()
for lag in (1, 2, 48, 336):
    df_feat[f"lag_{lag}"] = (
        df_feat.groupby("LCLid", observed=True)["energy"].shift(lag).astype("float32")
    )
print(f"  lags built in {time.time()-t0:.1f}s")

print("Building rolling features (may take 1-3 min)...")
t0 = time.time()
g = df_feat.groupby("LCLid", observed=True)["lag_1"]
for label, window, minp in [("1d", 48, 24), ("7d", 336, 168)]:
    df_feat[f"rolling_mean_{label}"] = (
        g.transform(lambda s, w=window, mp=minp: s.rolling(w, min_periods=mp).mean())
         .astype("float32")
    )
    df_feat[f"rolling_std_{label}"] = (
        g.transform(lambda s, w=window, mp=minp: s.rolling(w, min_periods=mp).std())
         .astype("float32")
    )
    print(f"  rolling_{label} done at {time.time()-t0:.1f}s")

before = len(df_feat)
df_feat = df_feat.dropna(
    subset=["lag_336", "rolling_mean_7d", "rolling_std_7d"]
).reset_index(drop=True)
print(f"\nDropped {before - len(df_feat):,} first-week-per-household rows "
      f"(lag_336 / rolling_7d NaN)")

f64 = df_feat.select_dtypes("float64").columns.tolist()
if f64:
    print(f"Downcasting float64 → float32: {f64}")
    for c in f64:
        df_feat[c] = df_feat[c].astype("float32")

def _save_features(df, cache_dir):
    import sys, subprocess
    try:
        import pyarrow  # noqa: F401
    except ImportError:
        print("pyarrow missing — attempting pip install (quiet)...")
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyarrow"])
            import pyarrow  # noqa: F401
            print("  pyarrow installed")
        except subprocess.CalledProcessError as e:
            print(f"  pyarrow install returned non-zero ({e.returncode}); trying fastparquet...")
        except Exception as e:
            print(f"  pyarrow install raised {type(e).__name__}: {e}")
            print("  trying fastparquet...")
    # try pyarrow
    try:
        p = cache_dir / "features_sample500.parquet"
        df.to_parquet(p, engine="pyarrow", compression="zstd", index=False)
        return p
    except Exception as e:
        print(f"pyarrow write failed: {e}")
    # try fastparquet
    try:
        import fastparquet  # noqa: F401
        p = cache_dir / "features_sample500.parquet"
        df.to_parquet(p, engine="fastparquet", compression="snappy", index=False)
        return p
    except Exception as e:
        print(f"fastparquet path failed: {e}")
    # final fallback: pickle (gzip uses stdlib only; no extra dep)
    p = cache_dir / "features_sample500.pkl.gz"
    try:
        df.to_pickle(p, compression="gzip")
    except Exception as e:
        print(f"pickle+gzip failed: {e}; writing raw pickle")
        p = cache_dir / "features_sample500.pkl"
        df.to_pickle(p)
    print(f"Fell back to pickle: {p.name}")
    return p

parquet_path = _save_features(df_feat, CACHE_DIR)
sz_mb = parquet_path.stat().st_size / 1e6
print(f"\nSaved {parquet_path.name}  ({sz_mb:,.1f} MB)")

print(f"\ndf_feat final shape: {df_feat.shape}")
print(f"memory: {df_feat.memory_usage(deep=True).sum()/1e6:,.1f} MB")
na_summary = df_feat.isna().sum()
print("\nNaN per column:")
print(na_summary[na_summary > 0].to_string() if na_summary.sum() > 0 else "  (none)")

gc.collect()
df_feat.head()

### 5.5 Temporal split + metric function

Split globally by `tstp` using the **same cutoffs `train_cutoff` / `val_cutoff` computed in 5.4** (so `household_train_mean` and the split agree to the row). No per-household split: each `LCLid` appears in all three buckets, just at different times.

- `train_mask` : `tstp <  train_cutoff`  (~70 %)
- `val_mask`   : `train_cutoff ≤ tstp < val_cutoff`  (~15 %)
- `test_mask`  : `tstp ≥ val_cutoff`  (~15 %)

**Sequence rule for 5.8** — an LSTM window is assigned to train / val / test by the **target timestamp `t`**, not by the window start. This keeps the benchmark aligned with the tabular models while still letting the network consume the recent history immediately before `t`.

**Seasonal caveat** — with data ending 2014-02-27, the last 15 % ≈ the last **winter months**. Head-line numbers will therefore reflect winter-heavy conditions; the 5.9 bucketed RMSE by hour/month/group is the proper diagnostic.

Metric `score(y_true, y_pred, name)` returns `{mae, rmse, bias, mape_masked_%}`. `MAPE` is only computed on rows with `y_true ≥ 0.05 kWh` (typical half-hour baseline for an occupied home) to avoid the usual near-zero blow-up.


In [ ]:
train_mask = df_feat["tstp"] <  train_cutoff
val_mask   = (df_feat["tstp"] >= train_cutoff) & (df_feat["tstp"] < val_cutoff)
test_mask  = df_feat["tstp"] >= val_cutoff

n_total = len(df_feat)
for name, m in [("train", train_mask), ("val", val_mask), ("test", test_mask)]:
    n = int(m.sum())
    pct = 100 * n / n_total
    t_lo = df_feat.loc[m, "tstp"].min()
    t_hi = df_feat.loc[m, "tstp"].max()
    hh   = df_feat.loc[m, "LCLid"].nunique()
    print(f"{name:5s}: n = {n:>12,}  ({pct:5.2f}%)  LCLids = {hh:>3d}  "
          f"range {t_lo} → {t_hi}")

assert int(train_mask.sum() + val_mask.sum() + test_mask.sum()) == n_total, "masks don't cover df_feat"
assert int((train_mask & val_mask ).sum()) == 0, "train/val overlap"
assert int((val_mask   & test_mask).sum()) == 0, "val/test overlap"
assert int((train_mask & test_mask).sum()) == 0, "train/test overlap"
print("\nMasks form a clean partition of df_feat.")

def score(y_true, y_pred, name: str = "") -> dict:
    y_true = np.asarray(y_true, dtype=np.float32)
    y_pred = np.asarray(y_pred, dtype=np.float32)
    resid = y_pred - y_true
    mae   = float(np.mean(np.abs(resid)))
    rmse  = float(np.sqrt(np.mean(resid ** 2)))
    bias  = float(np.mean(resid))
    m     = y_true >= 0.05
    mape  = float(np.mean(np.abs(resid[m] / y_true[m])) * 100) if m.any() else float("nan")
    return {"model": name, "n": int(len(y_true)),
            "mae": mae, "rmse": rmse, "bias": bias, "mape_masked_%": mape}

_y = df_feat.loc[train_mask, "energy"].to_numpy()
_s_perfect = score(_y, _y, "sanity_perfect")
assert _s_perfect["mae"] == 0.0 and _s_perfect["rmse"] == 0.0 and _s_perfect["bias"] == 0.0
_mean_pred = np.full_like(_y, _y.mean(), dtype=np.float32)
_s_const = score(_y, _mean_pred, "sanity_mean_pred")
print("\nscore() self-tests:")
print(" ", _s_perfect)
print(" ", _s_const)
print(f"\nScore helper and masks are ready. "
      f"Use df_feat.loc[train_mask|val_mask|test_mask, ...] in 5.6/5.7.")

### 5.6 Baselines — Seasonal naive + Ridge on `log1p(y)`

Two references that LightGBM in 5.7 has to beat.

**Seasonal naive** (`ŷ = lag_336`, i.e. the reading one week earlier at the same half-hour). This is the strongest no-parameters benchmark for half-hourly household load — captures weekly rhythm, daily shape, and household identity implicitly. No training needed.

**Ridge on `log1p(y)`**: linear with L2 regularisation, trained to predict `log1p(energy)` then inverted via `expm1` and clipped at 0. `log1p` tames the right-skew of half-hourly kWh (lots of zeros, a few bursts). Features:

- **Numeric** (21 cols, median-imputed + `RobustScaler`): weather + `hdd/cdd`, cyclic encodings (`hh_sin/cos`, `dow_sin/cos`), `is_holiday/is_weekend`, `lag_1/48/336`, `rolling_mean_1d/7d`, `household_train_mean`.
- **Categorical** (4 cols, one-hot): `Acorn_grouped`, `stdorToU`, `precipType`, `icon`.

**Training set**: uniformly subsampled 25 % of train rows (≈ 2.5 M) for a fast, memory-friendly fit. `solver="sparse_cg"` avoids materialising a dense Gram matrix.

**Reporting** — val + test scores for both models. Per 5.5, **val is "monitoring only"; only test numbers go in the write-up**.

In [ ]:
import time
from sklearn.linear_model import Ridge
from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

results = []

print("--- Seasonal naive (ŷ = lag_336) ---")
for split_name, m in [("val", val_mask), ("test", test_mask)]:
    y_true = df_feat.loc[m, "energy"].to_numpy()
    y_pred = df_feat.loc[m, "lag_336"].to_numpy()
    s = {**score(y_true, y_pred, "seasonal_naive"), "split": split_name}
    print(f"  {split_name:4s}: MAE={s['mae']:.4f}  RMSE={s['rmse']:.4f}  bias={s['bias']:+.4f}")
    results.append(s)

NUM_FEATS = [
    "temperature", "apparentTemperature", "humidity", "windSpeed",
    "pressure", "dewPoint", "visibility", "hdd", "cdd",
    "hh_sin", "hh_cos", "dow_sin", "dow_cos",
    "is_holiday", "is_weekend",
    "lag_1", "lag_48", "lag_336",
    "rolling_mean_1d", "rolling_mean_7d",
    "household_train_mean",
]
CAT_FEATS = ["Acorn_grouped", "stdorToU", "precipType", "icon"]
ALL_FEATS = NUM_FEATS + CAT_FEATS

X_all = df_feat[ALL_FEATS].copy()
for c in CAT_FEATS:
    if X_all[c].isna().any():
        X_all[c] = X_all[c].astype(str).fillna("missing")

RIDGE_SUBSAMPLE = 0.25
train_idx_all = np.flatnonzero(train_mask.to_numpy())
rng = np.random.default_rng(RANDOM_STATE)
sub_idx = rng.choice(train_idx_all, size=int(len(train_idx_all) * RIDGE_SUBSAMPLE), replace=False)
sub_idx.sort()

X_train = X_all.iloc[sub_idx]
y_train = df_feat["energy"].iloc[sub_idx].to_numpy().astype("float32")
y_train_log = np.log1p(y_train)
print(f"\n--- Ridge on log1p(y) ---")
print(f"Training rows: {len(X_train):,}  (subsample={RIDGE_SUBSAMPLE:.0%} of {len(train_idx_all):,})")

ct = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imp", SimpleImputer(strategy="median")),
            ("scale", RobustScaler()),
        ]), NUM_FEATS),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=True), CAT_FEATS),
    ],
    sparse_threshold=0.3,
)
ridge_pipe = Pipeline([
    ("ct", ct),
    ("ridge", Ridge(alpha=1.0, solver="sparse_cg", random_state=RANDOM_STATE)),
])

t0 = time.time()
ridge_pipe.fit(X_train, y_train_log)
print(f"  fit:     {time.time()-t0:.1f}s")

for split_name, m in [("val", val_mask), ("test", test_mask)]:
    idx = np.flatnonzero(m.to_numpy())
    X = X_all.iloc[idx]
    y_true = df_feat["energy"].iloc[idx].to_numpy().astype("float32")
    t0 = time.time()
    y_hat = np.clip(np.expm1(ridge_pipe.predict(X)), 0, None).astype("float32")
    elapsed = time.time() - t0
    s = {**score(y_true, y_hat, "ridge_log1p"), "split": split_name}
    print(f"  predict {split_name:4s}: {elapsed:4.1f}s  n={len(X):,}  "
          f"MAE={s['mae']:.4f}  RMSE={s['rmse']:.4f}  bias={s['bias']:+.4f}")
    results.append(s)

baseline_results = (
    pd.DataFrame(results)
      [["model", "split", "n", "mae", "rmse", "bias", "mape_masked_%"]]
      .sort_values(["split", "rmse"])
      .reset_index(drop=True)
)

print("\n=== Baseline scoreboard ===")
print(baseline_results.to_string(index=False))
print("\n(val is 'monitoring only' — only test numbers go in the write-up.)")

test_sn   = baseline_results.query("model=='seasonal_naive' and split=='test'").iloc[0]
test_ridge = baseline_results.query("model=='ridge_log1p'    and split=='test'").iloc[0]
rmse_improve_pct = 100 * (test_sn["rmse"] - test_ridge["rmse"]) / test_sn["rmse"]
mae_improve_pct  = 100 * (test_sn["mae"]  - test_ridge["mae"])  / test_sn["mae"]
print(f"\nOn test: Ridge beats seasonal naive by "
      f"{rmse_improve_pct:+.1f}% RMSE / {mae_improve_pct:+.1f}% MAE.")
print("LightGBM in 5.7 should beat Ridge by another decent margin to justify the complexity.")

### 5.7 LightGBM — pooled half-hourly regression

Boost on top of the baselines. Expected behaviour: clearly below Ridge on RMSE (Ridge is roughly linear-in-features; tree boosting should exploit the `lag × weather × time-of-day` interactions that Ridge can't express).

**Feature set (24 numeric + 5 categorical)**

- **Deliberately `LCLid` is NOT a feature.** If we gave the tree the household id it would memorise per-household residuals and the gain-importance plot would be uninformative. Instead `household_train_mean` (computed in 5.4 on the train window only) carries the "average household level" signal.
- **Numeric (24)**: all weather + `hdd/cdd`, time ints (`hour, half_hour, dow, month`), `is_holiday`, `is_weekend`, lags 1/2/48/336, rolling mean+std 1d/7d, `household_train_mean`. The cyclic `hh_sin/cos` + `dow_sin/cos` encodings are **skipped for trees** — integers already carry ordering; splits handle the seasonality without sinusoids.
- **Categorical (5, native LightGBM)**: `Acorn, Acorn_grouped, stdorToU, icon, precipType`.

**Params** (`regression_l1` objective, RMSE-monitored early stop):

```
learning_rate=0.025, num_leaves=95, min_data_in_leaf=500,
feature_fraction=0.85, bagging_fraction=0.85, bagging_freq=5,
lambda_l2=1.0, max_bin=255,
num_boost_round=6000, early_stopping=150 rounds on val RMSE, seed=42.
```

`regression_l1` is MAE-style — more robust to the right-skew of half-hourly kWh than squared error.

A first run at `lr=0.05, num_leaves=64, round=2000` got test RMSE ≈ 0.167 and was still descending at 1800 rounds (not converged). This config **halves the learning rate, doubles the budget, widens leaves slightly, tightens `min_data_in_leaf` + adds `lambda_l2`** to trade a bit of train fit for better val/test extrapolation (test is winter-heavy relative to train).

**Fallback** — if `lightgbm` refuses to install (Windows/Py 3.13), we fall back to `sklearn.ensemble.HistGradientBoostingRegressor` with `loss="absolute_error"` and categorical columns passed as `OrdinalEncoder` codes. Same spirit, slightly different knobs.

Artefacts saved under `models/`: `lgbm_hh.txt` (or `hgb_hh.joblib`) + a `*_features.json` with the exact column list used.

In [ ]:
import sys, subprocess, time, gc, json

try:
    import lightgbm as lgb
    HAVE_LGB = True
    print(f"lightgbm available (v{lgb.__version__})")
except ImportError:
    print("lightgbm not installed — attempting pip install (quiet)...")
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "lightgbm"])
        import lightgbm as lgb
        HAVE_LGB = True
        print(f"  installed lightgbm v{lgb.__version__}")
    except Exception as e:
        print(f"  install failed: {type(e).__name__}: {e}")
        print("  falling back to HistGradientBoostingRegressor")
        HAVE_LGB = False

LGB_NUM_FEATS = [
    "temperature", "apparentTemperature", "humidity", "windSpeed",
    "pressure", "dewPoint", "visibility", "hdd", "cdd",
    "is_holiday", "is_weekend",
    "hour", "half_hour", "dow", "month",
    "lag_1", "lag_2", "lag_48", "lag_336",
    "rolling_mean_1d", "rolling_std_1d",
    "rolling_mean_7d", "rolling_std_7d",
    "household_train_mean",
]
LGB_CAT_FEATS = ["Acorn", "Acorn_grouped", "stdorToU", "icon", "precipType"]
LGB_ALL_FEATS = LGB_NUM_FEATS + LGB_CAT_FEATS

for c in LGB_CAT_FEATS:
    if df_feat[c].dtype.name != "category":
        df_feat[c] = df_feat[c].astype("category")

y_tr_np   = df_feat.loc[train_mask, "energy"].to_numpy().astype("float32")
y_val_np  = df_feat.loc[val_mask,   "energy"].to_numpy().astype("float32")
y_test_np = df_feat.loc[test_mask,  "energy"].to_numpy().astype("float32")

print(f"\nShapes  train={int(train_mask.sum()):>10,}"
      f"  val={int(val_mask.sum()):>10,}"
      f"  test={int(test_mask.sum()):>10,}"
      f"  |  features: {len(LGB_NUM_FEATS)} num + {len(LGB_CAT_FEATS)} cat")

lgb_results = []
lgb_model = None

if HAVE_LGB:
    X_tr   = df_feat.loc[train_mask, LGB_ALL_FEATS]
    X_val  = df_feat.loc[val_mask,   LGB_ALL_FEATS]
    X_test = df_feat.loc[test_mask,  LGB_ALL_FEATS]

    dtrain = lgb.Dataset(X_tr,  label=y_tr_np,  categorical_feature=LGB_CAT_FEATS, free_raw_data=True)
    dval   = lgb.Dataset(X_val, label=y_val_np, categorical_feature=LGB_CAT_FEATS, reference=dtrain, free_raw_data=False)

    params = {
        "objective": "regression_l1",
        "metric": "rmse",
        "learning_rate": 0.025,
        "num_leaves": 95,
        "min_data_in_leaf": 500,
        "feature_fraction": 0.85,
        "bagging_fraction": 0.85,
        "bagging_freq": 5,
        "lambda_l2": 1.0,
        "max_bin": 255,
        "seed": RANDOM_STATE,
        "verbose": -1,
        "num_threads": 0,
    }

    print("\n--- LightGBM training ---")
    t0 = time.time()
    lgb_model = lgb.train(
        params,
        dtrain,
        num_boost_round=6000,
        valid_sets=[dtrain, dval],
        valid_names=["train", "val"],
        callbacks=[
            lgb.early_stopping(stopping_rounds=150),
            lgb.log_evaluation(period=200),
        ],
    )
    fit_sec = time.time() - t0
    print(f"  fit: {fit_sec:.1f}s  |  best_iter = {lgb_model.best_iteration}  "
          f"|  best val RMSE = {lgb_model.best_score['val']['rmse']:.4f}")

    del X_tr, dtrain, dval
    gc.collect()

    for split_name, X, y in [("val", X_val, y_val_np), ("test", X_test, y_test_np)]:
        t0 = time.time()
        y_hat = np.clip(
            lgb_model.predict(X, num_iteration=lgb_model.best_iteration), 0, None
        ).astype("float32")
        elapsed = time.time() - t0
        s = {**score(y, y_hat, "lightgbm"), "split": split_name}
        print(f"  predict {split_name:4s}: {elapsed:4.1f}s  n={len(X):,}  "
              f"MAE={s['mae']:.4f}  RMSE={s['rmse']:.4f}  bias={s['bias']:+.4f}")
        lgb_results.append(s)

    model_path = MODEL_DIR / "lgbm_hh.txt"
    lgb_model.save_model(str(model_path), num_iteration=lgb_model.best_iteration)
    feats_path = MODEL_DIR / "lgbm_hh_features.json"
    feats_path.write_text(json.dumps({
        "num": LGB_NUM_FEATS, "cat": LGB_CAT_FEATS,
        "best_iteration": int(lgb_model.best_iteration),
    }, indent=2))
    print(f"\nSaved → {model_path.name} ({model_path.stat().st_size/1e6:.1f} MB), "
          f"{feats_path.name}")
    MODEL_TAG = "lightgbm"

else:
    from sklearn.ensemble import HistGradientBoostingRegressor
    import joblib

    X_all_boost = df_feat[LGB_ALL_FEATS].copy()
    for c in LGB_CAT_FEATS:
        X_all_boost[c] = X_all_boost[c].cat.codes.astype("int32")
    cat_idx = [LGB_ALL_FEATS.index(c) for c in LGB_CAT_FEATS]

    X_tr   = X_all_boost.loc[train_mask]
    X_val  = X_all_boost.loc[val_mask]
    X_test = X_all_boost.loc[test_mask]

    print("\n--- HistGradientBoostingRegressor (fallback) ---")
    lgb_model = HistGradientBoostingRegressor(
        loss="absolute_error",
        max_iter=800,
        learning_rate=0.05,
        max_leaf_nodes=64,
        min_samples_leaf=200,
        l2_regularization=0.0,
        early_stopping=True,
        n_iter_no_change=50,
        validation_fraction=0.1,
        random_state=RANDOM_STATE,
        categorical_features=cat_idx,
        verbose=1,
    )
    t0 = time.time()
    lgb_model.fit(X_tr, y_tr_np)
    print(f"  fit: {time.time()-t0:.1f}s  |  n_iter_ = {lgb_model.n_iter_}")

    del X_tr, X_all_boost
    gc.collect()

    for split_name, X, y in [("val", X_val, y_val_np), ("test", X_test, y_test_np)]:
        t0 = time.time()
        y_hat = np.clip(lgb_model.predict(X), 0, None).astype("float32")
        elapsed = time.time() - t0
        s = {**score(y, y_hat, "hist_gbr"), "split": split_name}
        print(f"  predict {split_name:4s}: {elapsed:4.1f}s  n={len(X):,}  "
              f"MAE={s['mae']:.4f}  RMSE={s['rmse']:.4f}  bias={s['bias']:+.4f}")
        lgb_results.append(s)

    model_path = MODEL_DIR / "hgb_hh.joblib"
    joblib.dump(lgb_model, model_path)
    feats_path = MODEL_DIR / "hgb_hh_features.json"
    feats_path.write_text(json.dumps({
        "num": LGB_NUM_FEATS, "cat": LGB_CAT_FEATS,
        "n_iter": int(lgb_model.n_iter_),
    }, indent=2))
    print(f"\nSaved → {model_path.name} ({model_path.stat().st_size/1e6:.1f} MB), "
          f"{feats_path.name}")
    MODEL_TAG = "hist_gbr"

del X_val, X_test
gc.collect()

lgb_df = pd.DataFrame(lgb_results)[["model", "split", "n", "mae", "rmse", "bias", "mape_masked_%"]]
all_results = (
    pd.concat([baseline_results, lgb_df], ignore_index=True)
      .sort_values(["split", "rmse"])
      .reset_index(drop=True)
)
print("\n=== Full scoreboard (seasonal naive vs Ridge vs boosting) ===")
print(all_results.to_string(index=False))

test_lgb   = lgb_df.query("split=='test'").iloc[0]
test_ridge = baseline_results.query("model=='ridge_log1p' and split=='test'").iloc[0]
test_sn    = baseline_results.query("model=='seasonal_naive' and split=='test'").iloc[0]
vs_ridge = 100 * (test_ridge["rmse"] - test_lgb["rmse"]) / test_ridge["rmse"]
vs_sn    = 100 * (test_sn["rmse"]    - test_lgb["rmse"]) / test_sn["rmse"]
print(f"\nOn test: {MODEL_TAG} beats Ridge by {vs_ridge:+.1f}% RMSE, "
      f"beats seasonal naive by {vs_sn:+.1f}% RMSE.")
print("(val numbers above are monitoring-only; test is the headline.)")


### 5.8 LSTM benchmark

Professor feedback suggested including a model built specifically for sequential data, so we add a compact **LSTM benchmark**.

Design choices:

- **Target remains the same**: next half-hour household consumption.
- **Sequence inputs**: the previous **96 half-hours (48 hours)** of dynamic features: recent `energy`, weather, holiday/weekend flags, and cyclic time encodings.
- **Static inputs**: `household_train_mean`, `Acorn_grouped`, and `stdorToU`.
- **Target-time split**: each window is assigned to train / val / test according to the timestamp being predicted, matching 5.5.
- **Compute guardrail**: train on a stratified benchmark subset of households and thin the training windows with a stride so the notebook still runs in Colab / desktop settings.
- **Comparison rule**: 5.9 will compare all models on the **common test subset** where every model has a prediction.


In [ ]:
import json, sys, subprocess, time, gc
from sklearn.model_selection import StratifiedShuffleSplit

try:
    import tensorflow as tf
    HAVE_TF = True
    print(f"tensorflow available (v{tf.__version__})")
except ImportError:
    print("tensorflow not installed — attempting pip install (quiet)...")
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "tensorflow"])
        import tensorflow as tf
        HAVE_TF = True
        print(f"  installed tensorflow v{tf.__version__}")
    except Exception as e:
        print(f"  install failed: {type(e).__name__}: {e}")
        HAVE_TF = False

LSTM_LOOKBACK = 96
LSTM_TRAIN_STRIDE = 8
LSTM_HOUSEHOLDS = 120
LSTM_BATCH_SIZE = 512
LSTM_EPOCHS = 12
LSTM_DYNAMIC_COLS = [
    "energy",
    "temperature", "apparentTemperature", "humidity", "windSpeed",
    "hdd", "cdd",
    "is_holiday", "is_weekend",
    "hh_sin", "hh_cos", "dow_sin", "dow_cos",
]
LSTM_STATIC_NUM_COLS = ["household_train_mean"]
LSTM_STATIC_CAT_COLS = ["Acorn_grouped", "stdorToU"]

lstm_results = []
lstm_pred_frames = []
lstm_context = {
    "lookback": LSTM_LOOKBACK,
    "train_stride": LSTM_TRAIN_STRIDE,
    "benchmark_households": 0,
}

if HAVE_TF:
    meta_lstm = (
        df_feat[["LCLid", "Acorn_grouped", "stdorToU"]]
        .drop_duplicates("LCLid")
        .reset_index(drop=True)
        .copy()
    )
    meta_lstm["stratum"] = meta_lstm["Acorn_grouped"].astype(str) + "|" + meta_lstm["stdorToU"].astype(str)
    stratum_counts = meta_lstm["stratum"].value_counts()
    tiny = stratum_counts[stratum_counts < 5].index.tolist()
    if tiny:
        print(f"Collapsing {len(tiny)} sparse LSTM strata into 'Other'")
        meta_lstm["stratum"] = meta_lstm["stratum"].where(~meta_lstm["stratum"].isin(tiny), "Other")

    stratum_counts = meta_lstm["stratum"].value_counts()
    n_classes = int(stratum_counts.shape[0])
    min_count = int(stratum_counts.min()) if len(stratum_counts) else 0

    n_lstm = min(LSTM_HOUSEHOLDS, len(meta_lstm))
    if n_lstm < len(meta_lstm):
        can_stratify = (min_count >= 2) and (n_lstm >= n_classes)
        if can_stratify:
            sss = StratifiedShuffleSplit(n_splits=1, train_size=n_lstm, random_state=RANDOM_STATE)
            idx_lstm, _ = next(sss.split(meta_lstm, meta_lstm["stratum"]))
            print(f"Using stratified LSTM household sample across {n_classes} strata")
        else:
            print(f"Falling back to random LSTM household sample (min stratum={min_count}, n_classes={n_classes}, sample={n_lstm})")
            rng = np.random.default_rng(RANDOM_STATE)
            idx_lstm = np.sort(rng.choice(len(meta_lstm), size=n_lstm, replace=False))
        lstm_ids = set(meta_lstm.iloc[idx_lstm]["LCLid"])
    else:
        lstm_ids = set(meta_lstm["LCLid"])

    df_lstm = (
        df_feat.loc[df_feat["LCLid"].isin(lstm_ids),
                    ["LCLid", "tstp", *LSTM_DYNAMIC_COLS, *LSTM_STATIC_NUM_COLS, *LSTM_STATIC_CAT_COLS]]
        .sort_values(["LCLid", "tstp"])
        .reset_index(drop=True)
        .copy()
    )
    lstm_context["benchmark_households"] = int(df_lstm["LCLid"].nunique())
    print(f"Benchmark households for LSTM: {lstm_context['benchmark_households']}  |  rows: {len(df_lstm):,}")

    train_mask_lstm = df_lstm["tstp"] < train_cutoff
    val_mask_lstm   = (df_lstm["tstp"] >= train_cutoff) & (df_lstm["tstp"] < val_cutoff)
    test_mask_lstm  = df_lstm["tstp"] >= val_cutoff

    dyn_fill = df_lstm.loc[train_mask_lstm, LSTM_DYNAMIC_COLS].median(numeric_only=True)
    df_lstm[LSTM_DYNAMIC_COLS] = df_lstm[LSTM_DYNAMIC_COLS].fillna(dyn_fill)

    dyn_mean = df_lstm.loc[train_mask_lstm, LSTM_DYNAMIC_COLS].mean(numeric_only=True)
    dyn_std  = df_lstm.loc[train_mask_lstm, LSTM_DYNAMIC_COLS].std(numeric_only=True).replace(0, 1)
    dyn_arr  = ((df_lstm[LSTM_DYNAMIC_COLS] - dyn_mean) / dyn_std).astype("float32").to_numpy()

    static_num = df_lstm[LSTM_STATIC_NUM_COLS].astype("float32")
    static_num_mean = static_num.loc[train_mask_lstm].mean(numeric_only=True)
    static_num_std  = static_num.loc[train_mask_lstm].std(numeric_only=True).replace(0, 1)
    static_num_scaled = ((static_num - static_num_mean) / static_num_std).astype("float32")

    static_cat = pd.get_dummies(
        df_lstm[LSTM_STATIC_CAT_COLS].astype(str),
        prefix=LSTM_STATIC_CAT_COLS,
        dtype="float32",
    )
    static_arr = pd.concat([static_num_scaled, static_cat], axis=1).to_numpy(dtype="float32")
    y_arr = df_lstm["energy"].astype("float32").to_numpy()

    halfhour_ns = 30 * 60 * 1_000_000_000
    ts_ns = df_lstm["tstp"].astype("int64").to_numpy()
    run_len = np.zeros(len(df_lstm), dtype=np.int32)
    for _, idx in df_lstm.groupby("LCLid", sort=False).indices.items():
        pos = np.asarray(idx)
        if len(pos) <= 1:
            continue
        streak = 0
        diffs_ok = np.diff(ts_ns[pos]) == halfhour_ns
        for j in range(1, len(pos)):
            if diffs_ok[j - 1]:
                streak += 1
            else:
                streak = 0
            run_len[pos[j]] = streak

    valid_targets = run_len >= LSTM_LOOKBACK
    idx_train_all = np.flatnonzero(valid_targets & train_mask_lstm.to_numpy())
    idx_val = np.flatnonzero(valid_targets & val_mask_lstm.to_numpy())
    idx_test = np.flatnonzero(valid_targets & test_mask_lstm.to_numpy())
    idx_train = idx_train_all[::LSTM_TRAIN_STRIDE]

    print(f"Valid target rows  train(all)={len(idx_train_all):,}  train(strided)={len(idx_train):,}  "
          f"val={len(idx_val):,}  test={len(idx_test):,}")

    class WindowSequence(tf.keras.utils.Sequence):
        def __init__(self, target_idx, batch_size, shuffle=False):
            self.target_idx = np.asarray(target_idx, dtype=np.int64)
            self.batch_size = int(batch_size)
            self.shuffle = bool(shuffle)
            self.order = np.arange(len(self.target_idx))
            self.on_epoch_end()

        def __len__(self):
            return int(np.ceil(len(self.target_idx) / self.batch_size))

        def __getitem__(self, batch_no):
            order_slice = self.order[batch_no * self.batch_size:(batch_no + 1) * self.batch_size]
            batch_targets = self.target_idx[order_slice]
            x_dyn = np.stack([dyn_arr[i - LSTM_LOOKBACK:i] for i in batch_targets], axis=0)
            x_static = static_arr[batch_targets]
            y_batch = y_arr[batch_targets]
            return {"dyn_input": x_dyn, "static_input": x_static}, y_batch

        def on_epoch_end(self):
            if self.shuffle and len(self.order) > 0:
                np.random.shuffle(self.order)

    train_seq = WindowSequence(idx_train, LSTM_BATCH_SIZE, shuffle=True)
    val_seq   = WindowSequence(idx_val,   LSTM_BATCH_SIZE, shuffle=False)
    test_seq  = WindowSequence(idx_test,  LSTM_BATCH_SIZE, shuffle=False)

    tf.keras.backend.clear_session()
    tf.random.set_seed(RANDOM_STATE)
    np.random.seed(RANDOM_STATE)

    dyn_input = tf.keras.Input(shape=(LSTM_LOOKBACK, len(LSTM_DYNAMIC_COLS)), name="dyn_input")
    x = tf.keras.layers.LSTM(64, name="encoder_lstm")(dyn_input)
    x = tf.keras.layers.Dropout(0.20)(x)

    static_input = tf.keras.Input(shape=(static_arr.shape[1],), name="static_input")
    z = tf.keras.layers.Dense(16, activation="relu", name="static_dense")(static_input)

    h = tf.keras.layers.Concatenate(name="fusion")([x, z])
    h = tf.keras.layers.Dense(32, activation="relu", name="fusion_dense")(h)
    h = tf.keras.layers.Dropout(0.10)(h)
    out = tf.keras.layers.Dense(1, name="energy_hat")(h)

    lstm_model = tf.keras.Model(inputs=[dyn_input, static_input], outputs=out, name="hh_lstm")
    lstm_model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss=tf.keras.losses.Huber(delta=0.10),
        metrics=[tf.keras.metrics.MeanAbsoluteError(name="mae")],
    )
    lstm_model.summary()

    callbacks = [
        tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-5),
    ]

    print("\n--- LSTM training ---")
    t0 = time.time()
    history = lstm_model.fit(
        train_seq,
        validation_data=val_seq,
        epochs=LSTM_EPOCHS,
        callbacks=callbacks,
        verbose=1,
    )
    fit_sec = time.time() - t0
    print(f"LSTM fit completed in {fit_sec:.1f}s  |  epochs run = {len(history.history['loss'])}")

    def collect_seq_predictions(seq, target_idx, split_name):
        pred = np.clip(lstm_model.predict(seq, verbose=1).reshape(-1), 0, None).astype("float32")
        y_true = y_arr[target_idx].astype("float32")
        pred_df = df_lstm.loc[target_idx, ["LCLid", "tstp"]].copy()
        pred_df["y_true"] = y_true
        pred_df["y_lstm"] = pred
        pred_df["split"] = split_name
        res = {**score(y_true, pred, "lstm"), "split": split_name}
        return res, pred_df

    for split_name, seq, idx_target in [("val", val_seq, idx_val), ("test", test_seq, idx_test)]:
        res, pred_df = collect_seq_predictions(seq, idx_target, split_name)
        lstm_results.append(res)
        lstm_pred_frames.append(pred_df)
        print(f"  {split_name:4s}: MAE={res['mae']:.4f}  RMSE={res['rmse']:.4f}  bias={res['bias']:+.4f}  n={res['n']:,}")

    lstm_model_path = MODEL_DIR / "lstm_hh.keras"
    lstm_model.save(lstm_model_path)
    lstm_meta_path = MODEL_DIR / "lstm_hh_features.json"
    lstm_meta_path.write_text(json.dumps({
        "lookback": LSTM_LOOKBACK,
        "train_stride": LSTM_TRAIN_STRIDE,
        "benchmark_households": lstm_context["benchmark_households"],
        "dynamic": LSTM_DYNAMIC_COLS,
        "static_num": LSTM_STATIC_NUM_COLS,
        "static_cat": LSTM_STATIC_CAT_COLS,
        "static_onehot_cols": list(static_num_scaled.columns) + list(static_cat.columns),
    }, indent=2))
    print(f"Saved → {lstm_model_path.name}, {lstm_meta_path.name}")

    del train_seq, val_seq, test_seq, dyn_arr, static_arr, y_arr
    gc.collect()

else:
    print("Skipping LSTM benchmark because tensorflow is unavailable.")

lstm_results_df = pd.DataFrame(lstm_results)
lstm_pred_df = (
    pd.concat(lstm_pred_frames, ignore_index=True)
    if lstm_pred_frames else
    pd.DataFrame(columns=["LCLid", "tstp", "y_true", "y_lstm", "split"])
)

if not lstm_results_df.empty:
    lstm_results_df = lstm_results_df[["model", "split", "n", "mae", "rmse", "bias", "mape_masked_%"]]
    print("\n=== LSTM benchmark scoreboard ===")
    print(lstm_results_df.to_string(index=False))
else:
    print("No LSTM benchmark results were produced.")


### 5.9 Evaluation & interpretation

Goal: turn the raw scoreboards into a comparison we can actually reason about.

1. **Full-panel tabular headline** — retain the original test table for seasonal naive / Ridge / boosting on the full 499-household panel.
2. **Common-subset headline including LSTM** — compare all models on the exact same `LCLid + tstp` rows where every model has a prediction.
3. **Predicted vs actual hexbin** — for the best model on the common subset.
4. **Bucketed RMSE** — by hour, month, `Acorn_grouped`, and `stdorToU` on that same common subset.
5. **LightGBM importance + SHAP** — keep the interpretable model view for “which drivers matter”.
6. **Takeaways + handoff** — connect Part 1's short-term demand model to Part 2's half-hourly carbon-intensity workflow.


In [ ]:
import matplotlib.pyplot as plt

headline_full = (
    all_results.query("split == 'test'")
               [["model", "n", "mae", "rmse", "bias", "mape_masked_%"]]
               .sort_values("rmse")
               .reset_index(drop=True)
               .rename(columns={"mape_masked_%": "mape_masked%"})
)
print("=== Full-panel test metrics (tabular models on all eligible households) ===")
print(headline_full.to_string(index=False, float_format=lambda v: f"{v:,.4f}"))

test_idx = np.flatnonzero(test_mask.to_numpy())
y_true_test = df_feat["energy"].iloc[test_idx].to_numpy().astype("float32")
y_sn = df_feat["lag_336"].iloc[test_idx].to_numpy().astype("float32")

X_test_ridge = X_all.iloc[test_idx]
y_ridge = np.clip(np.expm1(ridge_pipe.predict(X_test_ridge)), 0, None).astype("float32")
del X_test_ridge

if HAVE_LGB:
    X_test_boost = df_feat.loc[test_mask, LGB_ALL_FEATS]
    y_boost = np.clip(
        lgb_model.predict(X_test_boost, num_iteration=lgb_model.best_iteration),
        0, None,
    ).astype("float32")
else:
    X_test_boost = df_feat.loc[test_mask, LGB_ALL_FEATS].copy()
    for c in LGB_CAT_FEATS:
        X_test_boost[c] = X_test_boost[c].cat.codes.astype("int32")
    y_boost = np.clip(lgb_model.predict(X_test_boost), 0, None).astype("float32")

eval_df = pd.DataFrame({
    "tstp":          df_feat["tstp"].iloc[test_idx].to_numpy(),
    "LCLid":         df_feat["LCLid"].iloc[test_idx].to_numpy(),
    "hour":          df_feat["hour"].iloc[test_idx].to_numpy(),
    "month":         df_feat["month"].iloc[test_idx].to_numpy(),
    "Acorn_grouped": df_feat["Acorn_grouped"].iloc[test_idx].astype(str).to_numpy(),
    "stdorToU":      df_feat["stdorToU"].iloc[test_idx].astype(str).to_numpy(),
    "y_true":        y_true_test,
    "y_sn":          y_sn,
    "y_ridge":       y_ridge,
    "y_boost":       y_boost,
})
print(f"\nBase eval_df (test-only, no LSTM merge yet): {eval_df.shape}  "
      f"memory ≈ {eval_df.memory_usage(deep=True).sum()/1e6:.0f} MB")

MODEL_COLS_COMMON = [
    ("y_sn", "seasonal_naive", "#9e9e9e"),
    ("y_ridge", "ridge_log1p", "#1f77b4"),
    ("y_boost", MODEL_TAG, "#d62728"),
]

if not lstm_pred_df.empty:
    lstm_test_pred = (
        lstm_pred_df.query("split == 'test'")
                    [["LCLid", "tstp", "y_lstm"]]
                    .drop_duplicates(["LCLid", "tstp"])
    )
    eval_common = eval_df.merge(lstm_test_pred, on=["LCLid", "tstp"], how="inner")
    MODEL_COLS_COMMON.append(("y_lstm", "lstm", "#2ca02c"))
    print(f"Merged LSTM benchmark predictions: {eval_common.shape}  "
          f"benchmark households={lstm_context.get('benchmark_households', 0)}")
else:
    eval_common = eval_df.copy()
    print("LSTM benchmark unavailable; common subset falls back to tabular models only.")

required_pred_cols = [col for col, _, _ in MODEL_COLS_COMMON]
eval_common = eval_common.dropna(subset=["y_true", *required_pred_cols]).reset_index(drop=True)
print(f"Common-subset eval frame: {eval_common.shape}  "
      f"memory ≈ {eval_common.memory_usage(deep=True).sum()/1e6:.0f} MB")

headline_common = pd.DataFrame(
    [score(eval_common["y_true"], eval_common[col], label) for col, label, _ in MODEL_COLS_COMMON]
).sort_values("rmse").reset_index(drop=True)
headline_common = headline_common[["model", "n", "mae", "rmse", "bias", "mape_masked_%"]]
headline_common = headline_common.rename(columns={"mape_masked_%": "mape_masked%"})
print("\n=== Common-subset test metrics (same rows for every model below) ===")
print(headline_common.to_string(index=False, float_format=lambda v: f"{v:,.4f}"))

label_to_col = {label: col for col, label, _ in MODEL_COLS_COMMON}
best_common_model = headline_common.iloc[0]["model"]
best_common_col = label_to_col[best_common_model]
print(f"\nBest model on the common subset: {best_common_model}")

hi = float(np.quantile(eval_common["y_true"], 0.995))
fig, ax = plt.subplots(figsize=(7.5, 7))
hb = ax.hexbin(
    eval_common["y_true"],
    eval_common[best_common_col],
    gridsize=80,
    bins="log",
    cmap="viridis",
    mincnt=1,
    extent=(0, hi, 0, hi),
)
ax.plot([0, hi], [0, hi], ls="--", lw=1, color="white")
ax.set_xlim(0, hi)
ax.set_ylim(0, hi)
ax.set_aspect("equal", adjustable="box")
ax.set_box_aspect(1)
ax.set_xlabel("actual energy (kWh/hh)")
ax.set_ylabel(f"{best_common_model} predicted (kWh/hh)")
ax.set_title(
    f"{best_common_model} test predictions vs actuals  "
    f"(common subset, n={len(eval_common):,}, axis clipped at 99.5-pct = {hi:.2f})"
)
cb = fig.colorbar(hb, ax=ax, fraction=0.046, pad=0.04)
cb.set_label("log10(count per hexbin)")
plt.tight_layout()
plt.show()

del X_test_boost
gc.collect()


In [ ]:
def bucket_rmse(df: pd.DataFrame, by: str, model_cols) -> pd.DataFrame:
    rows = []
    yt = df["y_true"].to_numpy(dtype="float32")
    pred_arrays = {label: df[col].to_numpy(dtype="float32") for col, label, _ in model_cols}
    for key, idx in df.groupby(by, observed=True).indices.items():
        row = {"bucket": key, "n": len(idx)}
        y_sub = yt[idx]
        for _, label, _ in model_cols:
            resid = pred_arrays[label][idx] - y_sub
            row[label] = float(np.sqrt(np.mean(resid ** 2)))
        rows.append(row)
    return pd.DataFrame(rows)

sort_model = best_common_model if best_common_model in [label for _, label, _ in MODEL_COLS_COMMON] else MODEL_TAG
b_hour  = bucket_rmse(eval_common, "hour", MODEL_COLS_COMMON).sort_values("bucket").reset_index(drop=True)
b_month = bucket_rmse(eval_common, "month", MODEL_COLS_COMMON).sort_values("bucket").reset_index(drop=True)
b_acorn = bucket_rmse(eval_common, "Acorn_grouped", MODEL_COLS_COMMON).sort_values(sort_model).reset_index(drop=True)
b_tou   = bucket_rmse(eval_common, "stdorToU", MODEL_COLS_COMMON).sort_values(sort_model).reset_index(drop=True)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

ax = axes[0, 0]
for _, label, color in MODEL_COLS_COMMON:
    ax.plot(b_hour["bucket"], b_hour[label], marker="o", lw=1.5, color=color, label=label)
ax.set_xticks(range(0, 24, 2))
ax.set_xlabel("hour of day")
ax.set_ylabel("RMSE (kWh/hh)")
ax.set_title("Common-subset test RMSE by hour")
ax.legend(loc="upper left", fontsize=9)
ax.grid(alpha=0.3)

ax = axes[0, 1]
for _, label, color in MODEL_COLS_COMMON:
    ax.plot(b_month["bucket"], b_month[label], marker="o", lw=1.5, color=color, label=label)
ax.set_xlabel("month (test covers 2013-11 → 2014-02)")
ax.set_ylabel("RMSE (kWh/hh)")
ax.set_title("Common-subset test RMSE by month")
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

ax = axes[1, 0]
x = np.arange(len(b_acorn))
w = 0.80 / max(1, len(MODEL_COLS_COMMON))
for i, (_, label, color) in enumerate(MODEL_COLS_COMMON):
    ax.bar(x + (i - (len(MODEL_COLS_COMMON) - 1) / 2) * w, b_acorn[label], width=w, color=color, label=label)
ax.set_xticks(x)
ax.set_xticklabels(b_acorn["bucket"], rotation=15, ha="right")
ax.set_ylabel("RMSE (kWh/hh)")
ax.set_title("Common-subset test RMSE by Acorn_grouped")
ax.legend(fontsize=9)
ax.grid(axis="y", alpha=0.3)

ax = axes[1, 1]
x = np.arange(len(b_tou))
for i, (_, label, color) in enumerate(MODEL_COLS_COMMON):
    ax.bar(x + (i - (len(MODEL_COLS_COMMON) - 1) / 2) * w, b_tou[label], width=w, color=color, label=label)
ax.set_xticks(x)
ax.set_xticklabels(b_tou["bucket"])
ax.set_ylabel("RMSE (kWh/hh)")
ax.set_title("Common-subset test RMSE by stdorToU")
ax.legend(fontsize=9)
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

print("\n--- by hour (head/tail) ---")
print(b_hour.head(3).to_string(index=False, float_format=lambda v: f"{v:,.4f}"))
print("...")
print(b_hour.tail(3).to_string(index=False, float_format=lambda v: f"{v:,.4f}"))

print("\n--- by month ---")
print(b_month.to_string(index=False, float_format=lambda v: f"{v:,.4f}"))

print("\n--- by Acorn_grouped ---")
print(b_acorn.to_string(index=False, float_format=lambda v: f"{v:,.4f}"))

print("\n--- by stdorToU ---")
print(b_tou.to_string(index=False, float_format=lambda v: f"{v:,.4f}"))

if MODEL_TAG in b_tou.columns and set(b_tou["bucket"]) >= {"Std", "ToU"}:
    tou_gap_primary = float(
        b_tou.loc[b_tou["bucket"] == "ToU", MODEL_TAG].iloc[0]
        - b_tou.loc[b_tou["bucket"] == "Std", MODEL_TAG].iloc[0]
    )
    print(f"\nPrimary boosting-model RMSE gap (ToU − Std): {tou_gap_primary:+.4f} kWh/hh "
          f"({100 * tou_gap_primary / b_tou.loc[b_tou['bucket']=='Std', MODEL_TAG].iloc[0]:+.1f}%)")
else:
    tou_gap_primary = float('nan')


In [ ]:
import sys, subprocess, time, gc

if not HAVE_LGB or lgb_model is None:
    print("LightGBM-specific interpretation is unavailable because the fallback hist_gbr model was used.")
else:
    imp = pd.DataFrame({
        "feature": lgb_model.feature_name(),
        "gain":    lgb_model.feature_importance(importance_type="gain"),
        "split":   lgb_model.feature_importance(importance_type="split"),
    }).sort_values("gain", ascending=False).reset_index(drop=True)
    imp["gain_pct"]  = 100 * imp["gain"]  / imp["gain"].sum()
    imp["split_pct"] = 100 * imp["split"] / imp["split"].sum()

    print("=== LightGBM importance (top 20 by gain) ===")
    print(imp.head(20)[["feature", "gain", "gain_pct", "split", "split_pct"]]
             .to_string(index=False, float_format=lambda v: f"{v:,.2f}"))

    top20 = imp.head(20).iloc[::-1]
    fig, ax = plt.subplots(figsize=(8.5, 7.5))
    ax.barh(top20["feature"], top20["gain_pct"], color="#d62728")
    ax.set_xlabel("gain share (%)")
    ax.set_title(f"LightGBM gain importance — top 20 of {len(imp)}")
    for y_pos, (v, s) in enumerate(zip(top20["gain_pct"], top20["split"])):
        ax.text(v + 0.1, y_pos, f"{v:,.1f}%  (splits={s:,})", va="center", fontsize=8)
    ax.grid(axis="x", alpha=0.3)
    plt.tight_layout()
    plt.show()

    print("\n--- SHAP summary on 20k-row test subsample ---")
    try:
        import shap
    except ImportError:
        print("  shap not installed — attempting pip install (quiet)...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "shap"])
        import shap
    print(f"  shap v{shap.__version__}")

    X_shap_source = df_feat.loc[test_mask, LGB_ALL_FEATS].copy()
    rng = np.random.default_rng(RANDOM_STATE)
    n_shap = min(20000, len(X_shap_source))
    shap_idx = rng.choice(len(X_shap_source), size=n_shap, replace=False)
    X_shap = X_shap_source.iloc[shap_idx].reset_index(drop=True)

    t0 = time.time()
    explainer = shap.TreeExplainer(lgb_model)
    shap_values = explainer.shap_values(X_shap)
    shap_arr = np.asarray(shap_values)
    print(f"  SHAP values computed in {time.time()-t0:.1f}s, shape={shap_arr.shape}")

    X_shap_display = X_shap.copy()
    for c in LGB_CAT_FEATS:
        X_shap_display[c] = X_shap_display[c].cat.codes.astype("int32")

    mean_abs = pd.DataFrame({
        "feature": list(X_shap.columns),
        "mean_abs_shap": np.abs(shap_arr).mean(axis=0),
    }).sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)
    print("\n=== SHAP mean(|value|) top 15 ===")
    print(mean_abs.head(15).to_string(index=False, float_format=lambda v: f"{v:,.4f}"))

    plt.figure(figsize=(9, 7))
    shap.summary_plot(
        shap_values, X_shap_display,
        feature_names=list(X_shap.columns),
        max_display=20,
        show=False,
        plot_size=None,
    )
    plt.title(f"SHAP summary — LightGBM on {n_shap:,} test rows (cat features shown as integer codes)")
    plt.tight_layout()
    plt.show()

    del X_shap_source, X_shap, X_shap_display, shap_values, shap_arr
    gc.collect()


### 5.9.5 Takeaways & handoff to Part 2


In [ ]:
from IPython.display import Markdown, display

boost_full = headline_full.query("model == @MODEL_TAG").iloc[0]
ridge_full = headline_full.query("model == 'ridge_log1p'").iloc[0]
sn_full    = headline_full.query("model == 'seasonal_naive'").iloc[0]

summary_blocks = []
summary_blocks.append(
    "**Full-panel short-term headline.** On the full eligible test panel, "
    f"`{MODEL_TAG}` remains the strongest tabular model with MAE `{boost_full['mae']:.4f}` and RMSE `{boost_full['rmse']:.4f}`, "
    f"versus Ridge at RMSE `{ridge_full['rmse']:.4f}` and seasonal naive at RMSE `{sn_full['rmse']:.4f}`."
)

if "lstm" in set(headline_common["model"]):
    boost_common = headline_common.query("model == @MODEL_TAG").iloc[0]
    lstm_row = headline_common.query("model == 'lstm'").iloc[0]
    rmse_delta = 100 * (boost_common["rmse"] - lstm_row["rmse"]) / boost_common["rmse"]
    mae_delta  = 100 * (boost_common["mae"] - lstm_row["mae"]) / boost_common["mae"]
    if rmse_delta > 0:
        lstm_take = (
            f"On the common benchmark subset, `lstm` improves on `{MODEL_TAG}` by `{rmse_delta:.1f}%` RMSE "
            f"and `{mae_delta:.1f}%` MAE, so it becomes the strongest pure predictive model in that restricted comparison."
        )
    else:
        lstm_take = (
            f"On the common benchmark subset, `lstm` does not beat `{MODEL_TAG}` "
            f"(RMSE delta `{rmse_delta:.1f}%`, MAE delta `{mae_delta:.1f}%`), so it stays a useful sequence benchmark rather than the preferred production model."
        )
    summary_blocks.append("**LSTM benchmark decision.** " + lstm_take)
else:
    summary_blocks.append(
        "**LSTM benchmark decision.** TensorFlow/LSTM was not available at runtime, so the comparison remains between seasonal naive, Ridge, and the boosting model."
    )

if HAVE_LGB and 'imp' in globals():
    top_feats = ", ".join(f"`{f}`" for f in imp.head(5)["feature"].tolist())
    summary_blocks.append(
        "**Driver story from the interpretable model.** LightGBM still provides the clearest explanation of short-term demand: "
        f"the top gain features are {top_feats}. That keeps the project aligned with the plan's requirement to quantify which drivers matter most."
    )

if not np.isnan(tou_gap_primary):
    summary_blocks.append(
        "**Tariff-group caveat.** On the common subset, the primary boosting model's `ToU − Std` RMSE gap is "
        f"`{tou_gap_primary:+.4f}` kWh/hh. This is the same pre-registered drift check as before, now reported on the fair four-model comparison subset."
    )

summary_blocks.append(
    "**Handoff to Part 2.** Part 1 stays focused on **short-term half-hourly electricity demand forecasting**, which aligns directly with the half-hourly carbon-intensity series used downstream. "
    "Longer-term carbon impact (daily / monthly / seasonal aggregation) belongs in Part 2, where predicted or observed half-hourly demand can be aggregated after the emissions join."
)

display(Markdown("\n\n".join(summary_blocks)))
